In [1]:
import pandas as pd
import numpy as np

In [3]:
# Load cleaned listings
listings = pd.read_csv("../datasets/processed/new_york_cleaned.csv")

# Load calendar dataset
calendar = pd.read_csv("../datasets/raw/New York calendar.csv")

# Load reviews dataset
reviews = pd.read_csv("../datasets/raw/New York reviews.csv")

In [4]:
listings.head()

,id,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bedrooms,beds,price,availability_30,availability_60,availability_90,availability_365,number_of_reviews,review_scores_rating,reviews_per_month,estimated_occupancy_l365d,estimated_revenue_l365d,host_is_superhost
0,2539,Kensington,40.645937,-73.972164,Private room in condo,Private room,2,1.0,2.0,113.97,4,34,64,339,10,4.80,0.08,60,6838.0,No
1,6848,Williamsburg,40.709350,-73.953420,Entire rental unit,Entire home/apt,3,2.0,1.0,117.27,1,1,15,191,198,4.60,0.95,180,21109.0,Yes
2,6872,East Harlem,40.801070,-73.942550,Private room in condo,Private room,1,1.0,1.0,80.06,0,0,11,286,2,5.00,0.04,60,4804.0,No
3,6990,East Harlem,40.787780,-73.947590,Private room in rental unit,Private room,1,1.0,1.0,77.17,0,28,28,218,251,4.88,1.24,120,9260.0,Yes
4,7097,Fort Greene,40.691940,-73.973890,Private room in guest suite,Private room,2,1.0,2.0,202.47,0,0,0,106,423,4.89,2.12,255,51630.0,Yes


In [5]:
calendar.head()

,listing_id,date,available,minimum_nights,maximum_nights
0,115535,2026-06-14,f,30,365
1,115535,2026-06-15,t,30,365
2,115535,2026-06-16,f,30,365
3,115535,2026-06-17,f,30,365
4,115535,2026-06-18,f,30,365


In [6]:
reviews.head()

,listing_id,id,date,reviewer_id,reviewer_name,comments
0,2539,55688172,2015-12-04,25160947,Peter,Great host
1,2539,97474898,2016-08-27,91513326,Liz,Nice room for the price. Great neighborhood. J...
2,2539,105340344,2016-10-01,90022459,Евгений,Very nice apt. New remodeled.
3,2539,133131670,2017-02-20,116165195,George,Great place to stay for a while. John is a gre...
4,2539,138349776,2017-03-19,118432644,Carlos,.


In [7]:
listings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30259 entries, 0 to 30258
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         30259 non-null  int64  
 1   neighbourhood_cleansed     30259 non-null  object 
 2   latitude                   30259 non-null  float64
 3   longitude                  30259 non-null  float64
 4   property_type              30259 non-null  object 
 5   room_type                  30259 non-null  object 
 6   accommodates               30259 non-null  int64  
 7   bedrooms                   30259 non-null  float64
 8   beds                       30259 non-null  float64
 9   price                      30259 non-null  float64
 10  availability_30            30259 non-null  int64  
 11  availability_60            30259 non-null  int64  
 12  availability_90            30259 non-null  int64  
 13  availability_365           30259 non-null  int

In [8]:
calendar.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11152576 entries, 0 to 11152575
Data columns (total 5 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   listing_id      int64 
 1   date            object
 2   available       object
 3   minimum_nights  int64 
 4   maximum_nights  int64 
dtypes: int64(3), object(2)
memory usage: 425.4+ MB


In [9]:
reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 990170 entries, 0 to 990169
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   listing_id     990170 non-null  int64 
 1   id             990170 non-null  int64 
 2   date           990170 non-null  object
 3   reviewer_id    990170 non-null  int64 
 4   reviewer_name  990167 non-null  object
 5   comments       989898 non-null  object
dtypes: int64(3), object(3)
memory usage: 45.3+ MB


In [10]:
calendar.columns

Index(['listing_id', 'date', 'available', 'minimum_nights', 'maximum_nights'], dtype='object')

In [11]:
# Convert date to datetime
calendar["date"] = pd.to_datetime(calendar["date"])

# Month number
calendar["Month_Number"] = calendar["date"].dt.month

# Month name
calendar["Month"] = calendar["date"].dt.month_name()

# Year
calendar["Year"] = calendar["date"].dt.year

# Season
def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"

calendar["Season"] = calendar["Month_Number"].apply(get_season)

In [12]:
calendar[["date", "Month", "Season"]].head()

,date,Month,Season
0,2026-06-14,June,Summer
1,2026-06-15,June,Summer
2,2026-06-16,June,Summer
3,2026-06-17,June,Summer
4,2026-06-18,June,Summer


In [13]:
# Convert review date to datetime
reviews["date"] = pd.to_datetime(reviews["date"])

# Month Number
reviews["Month_Number"] = reviews["date"].dt.month

# Month Name
reviews["Month"] = reviews["date"].dt.month_name()

# Year
reviews["Year"] = reviews["date"].dt.year

# Season
reviews["Season"] = reviews["Month_Number"].apply(get_season)

In [14]:
reviews[["date","Month","Season"]].head()

,date,Month,Season
0,2015-12-04,December,Winter
1,2016-08-27,August,Summer
2,2016-10-01,October,Autumn
3,2017-02-20,February,Winter
4,2017-03-19,March,Spring


# Seasonal Availability Analysis

In [15]:
seasonal_availability = (
    calendar.groupby("Season")
    .agg(
        Total_Days=("available", "count"),
        Available_Days=("available", lambda x: (x == "t").sum()),
        Booked_Days=("available", lambda x: (x == "f").sum())
    )
    .reset_index()
)

seasonal_availability["Occupancy_Rate (%)"] = (
    seasonal_availability["Booked_Days"] /
    seasonal_availability["Total_Days"] * 100
).round(2)

seasonal_availability

,Season,Total_Days,Available_Days,Booked_Days,Occupancy_Rate (%)
0,Autumn,2780505,1560489,1220016,43.88
1,Spring,2811060,1382021,1429039,50.84
2,Summer,2811061,1220943,1590118,56.57
3,Winter,2749950,1544334,1205616,43.84


In [16]:
monthly_availability = (
    calendar.groupby("Month")
    .agg(
        Total_Days=("available","count"),
        Available_Days=("available",lambda x:(x=="t").sum()),
        Booked_Days=("available",lambda x:(x=="f").sum())
    )
    .reset_index()
)

monthly_availability["Occupancy_Rate (%)"] = (
    monthly_availability["Booked_Days"] /
    monthly_availability["Total_Days"] * 100
).round(2)

monthly_availability

,Month,Total_Days,Available_Days,Booked_Days,Occupancy_Rate (%)
0,April,916650,443442,473208,51.62
1,August,947205,505570,441635,46.63
2,December,947205,522360,424845,44.85
3,February,855540,486975,368565,43.08
4,January,947205,534999,412206,43.52
5,July,947205,379024,568181,59.99
6,June,916651,336349,580302,63.31
7,March,947205,490365,456840,48.23
8,May,947205,448214,498991,52.68
9,November,916650,536468,380182,41.48


In [17]:
seasonal_reviews = (
    reviews.groupby("Season")
    .size()
    .reset_index(name="Total_Reviews")
)

seasonal_reviews

,Season,Total_Reviews
0,Autumn,273870
1,Spring,255182
2,Summer,261990
3,Winter,199128


In [18]:
monthly_reviews = (
    reviews.groupby("Month")
    .size()
    .reset_index(name="Total_Reviews")
)

monthly_reviews

,Month,Total_Reviews
0,April,81860
1,August,90797
2,December,83089
3,February,52597
4,January,63442
5,July,84063
6,June,87130
7,March,72031
8,May,101291
9,November,80211


In [19]:
seasonal_availability.to_csv("../datasets/processed/seasonal_availability.csv", index=False)

monthly_availability.to_csv("../datasets/processed/monthly_availability.csv", index=False)

seasonal_reviews.to_csv("../datasets/processed/seasonal_reviews.csv", index=False)

monthly_reviews.to_csv("../datasets/processed/monthly_reviews.csv", index=False)

In [20]:
month_order = {
    "January":1,
    "February":2,
    "March":3,
    "April":4,
    "May":5,
    "June":6,
    "July":7,
    "August":8,
    "September":9,
    "October":10,
    "November":11,
    "December":12
}

monthly_availability["Month_Number"] = monthly_availability["Month"].map(month_order)

monthly_availability = monthly_availability.sort_values("Month_Number")

monthly_availability

,Month,Total_Days,Available_Days,Booked_Days,Occupancy_Rate (%),Month_Number
4,January,947205,534999,412206,43.52,1
3,February,855540,486975,368565,43.08,2
7,March,947205,490365,456840,48.23,3
0,April,916650,443442,473208,51.62,4
8,May,947205,448214,498991,52.68,5
6,June,916651,336349,580302,63.31,6
5,July,947205,379024,568181,59.99,7
1,August,947205,505570,441635,46.63,8
11,September,916650,499224,417426,45.54,9
10,October,947205,524797,422408,44.60,10


In [21]:
monthly_reviews["Month_Number"] = monthly_reviews["Month"].map(month_order)

monthly_reviews = monthly_reviews.sort_values("Month_Number")

monthly_reviews

,Month,Total_Reviews,Month_Number
4,January,63442,1
3,February,52597,2
7,March,72031,3
0,April,81860,4
8,May,101291,5
6,June,87130,6
5,July,84063,7
1,August,90797,8
11,September,97368,9
10,October,96291,10


In [22]:
monthly_availability.to_csv(
    "../datasets/processed/monthly_availability.csv",
    index=False
)

monthly_reviews.to_csv(
    "../datasets/processed/monthly_reviews.csv",
    index=False
)